# 무한 개소리 생성기 — Google Colab (T4 GPU) 버전

Gemma 3 **사전학습(base, `-pt`)** 모델 기반의 무한 개소리 생성기입니다.

로컬 구동용 `gomi.py` 를 Colab T4 환경에 맞게 포팅한 버전입니다.

## 동작 원리
1. Gemma 사전학습(base, `-pt`) 모델을 로드 (instruct 아님 = 문장 이어붙이기만 함)
2. 랜덤 단어를 마구 섞어 "문장 아닌 문장"을 시드로 입력
3. 높은 temperature 로 출력
4. 출력에서 단어 몇 개를 랜덤으로 뽑아 다음 입력으로 사용 (이전 컨텍스트는 버린다. 매 라운드 독립)
5. 무한 반복 (Colab 의 ■ 중지 버튼으로 종료)

## Colab 사용 방법
1. 상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 선택
2. 아래 셀을 **위에서부터 순서대로** 실행
3. Gemma 모델은 라이선스 동의 + HF 토큰이 필요합니다 (아래 안내 참고)

## T4 관련 참고
- T4 (Turing, compute 7.5) 는 bf16 을 네이티브로 지원하지 않으므로 **fp16** 을 사용합니다.
- VRAM 약 15GB → `gemma-3-1b-pt` (fp16 약 2GB) 는 여유롭게 구동됩니다.
- 더 큰 `gemma-3-4b-pt` 는 `USE_4BIT = True` 로 켜면 됩니다 (bitsandbytes 4bit).

## 1. 의존성 설치
Colab 에는 torch 가 이미 깔려 있으므로 transformers 계열만 최신화합니다.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    cc = torch.cuda.get_device_capability(0)
    print("Compute capability:", cc)
    print("bf16 지원:", torch.cuda.is_bf16_supported())

## 2. Hugging Face 로그인 (Gemma 라이선스 동의 필요)

Gemma 3 모델은 다운로드 전에 약관 동의와 토큰 인증이 필요합니다.
1. https://huggingface.co/google/gemma-3-1b-pt 에서 **Acknowledge license** (동의)
2. https://huggingface.co/settings/tokens 에서 **Read** 권한 토큰 생성/복사

**권장 방법:** Colab 좌측 🔑(열쇠) 아이콘 → `HF_TOKEN` 이름으로 비밀(Secret) 추가 → 노트북 액세스 허용.
비밀을 설정하지 않았다면 아래 실행 시 토큰을 직접 입력하는 칸이 나타납니다.

In [ ]:
import os
from huggingface_hub import login

hf_token = None

# 1) Colab Secrets(userdata) 에 HF_TOKEN 이 있으면 우선 사용
try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

# 2) 환경 변수
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

# 3) 둘 다 없으면 직접 입력 받기 (입력 숨김)
if not hf_token:
    from getpass import getpass
    hf_token = getpass("HF_TOKEN 입력 (Read 권한 토큰): ").strip()

if hf_token:
    login(token=hf_token)
    print("✅ Hugging Face 로그인 완료")
else:
    print("⚠️  토큰이 없습니다. 게이트가 걸린 Gemma 모델은 로드되지 않을 수 있습니다.")

## 3. 설정
취향에 맞게 값을 바꾼 뒤 실행하세요.

In [ ]:
# ============================ 설정 ============================
MODEL_ID   = "google/gemma-3-1b-pt"   # 반드시 base(-pt)! (4b 쓰려면 "google/gemma-3-4b-pt")
USE_4BIT   = False    # True 로 켜면 더 큰 모델(예: gemma-3-4b-pt)을 4bit 로 로드

SEED_WORDS     = 7     # 시작 시드에 쓸 단어 수
FEEDBACK_WORDS = 4     # 출력에서 다음 입력으로 재활용할 단어 수
MAX_NEW_TOKENS = 256   # 한 라운드에 생성할 토큰 수

# 동적 온도 조절 (Dynamic Temperature)
USE_DYNAMIC_TEMP    = True   # True: 루프마다 온도를 사인파로 변동, False: 고정 온도
DYNAMIC_TEMP_MIN    = 1.2    # 최소 온도 (상대적으로 정상적인 출력)
DYNAMIC_TEMP_MAX    = 2.0    # 최대 온도 (광기 어린 무작위 출력)
DYNAMIC_TEMP_PERIOD = 10     # 사인파의 한 주기(주기당 라운드 수)
SHOW_TEMP_INDICATOR = True   # 각 라운드 앞에 현재 온도를 표시할지 여부

TEMPERATURE        = 1.7   # USE_DYNAMIC_TEMP=False 일 때 쓸 고정 온도 (1.3 ~ 2.0 추천)
TOP_P              = 0.97
TOP_K              = 0     # 0 = 비활성(완전 카오스). 출력이 깨지면 80~120 으로
REPETITION_PENALTY = 1.15  # 같은 단어 무한반복 방지

SLEEP_BETWEEN = 0.4   # 라운드 사이 쉬는 시간(초). 0 이면 쉬지 않음

# Colab 은 키보드 ESC/일시정지 핸들링이 어려우므로 라운드 수로 제어합니다.
# 0 이하 = 무한 반복 (Colab ■ 중지 버튼으로 종료)
MAX_ROUNDS = 0
# =============================================================
print("설정 완료")

## 4. 단어 풀 & 유틸 함수

In [ ]:
import re
import time
import random
import math

# ---------- 랜덤 단어 풀 ----------
DEFAULT_WORDS = """
banana velvet thunder marble whisper engine pickle galaxy ribbon cactus
mirror plastic comet jelly anchor lantern fossil noodle pebble syrup
quantum carpet dragon biscuit magnet violin orbit pretzel feather kettle
hollow trumpet glacier muffin spiral walnut cobweb turbine pixel mango
shadow bucket crayon tundra zipper lemon goblin yacht wrinkle tofu
허무 바나나 구름 망치 고양이 우주 양말 라면 번개 거울
""".split()

def load_wordbank():
    """시스템 사전이 있으면 그걸 쓰고, 없으면 기본 단어 풀 사용."""
    for path in ("/usr/share/dict/words",):
        if os.path.exists(path):
            try:
                with open(path, encoding="utf-8", errors="ignore") as f:
                    w = [ln.strip() for ln in f
                         if ln.strip().isalpha() and 3 <= len(ln.strip()) <= 11]
                if len(w) > 500:
                    return w
            except Exception:
                pass
    return DEFAULT_WORDS

WORDBANK = load_wordbank()

def random_seed(n):
    """랜덤 단어 n개를 섞어 '문장 아닌 문장'을 만든다."""
    return " ".join(random.sample(WORDBANK, k=min(n, len(WORDBANK))))

def pick_words(text, n):
    """출력 텍스트에서 단어 n개를 랜덤으로 뽑는다. 비면 새 시드로 폴백."""
    words = re.findall(r"\w+", text)          # 유니코드 단어(한/영/숫자) 매칭
    if not words:
        return random_seed(n)
    random.shuffle(words)
    return " ".join(words[:n])

print(f"단어 풀 크기: {len(WORDBANK)}")

## 5. 모델 로드 (T4: fp16)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer

def load_model():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("⚠️  CUDA를 못 찾았습니다. CPU로 돌아가지만 매우 느립니다.")
        print("    Colab 상단 [런타임 → 런타임 유형 변경 → T4 GPU] 를 확인하세요.")

    # T4(Turing) 는 bf16 비네이티브 → fp16 사용. 그 외 지원 GPU 면 bf16.
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
    print(f"모델 로딩 중: {MODEL_ID}  (device={device}, dtype={compute_dtype}) ...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    kwargs = dict(torch_dtype=compute_dtype)
    if USE_4BIT:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_quant_type="nf4",
        )
        kwargs["device_map"] = "cuda"

    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
    if not USE_4BIT:
        model.to(device)
    model.eval()
    print("✅ 로딩 완료.\n")
    return tokenizer, model

tokenizer, model = load_model()

## 6. 생성 함수
Colab 셀 출력에 맞춰 줄바꿈을 공백으로 바꿔 한 줄로 흐르게 합니다.

In [ ]:
class NoNewlineStreamer(TextStreamer):
    """줄바꿈을 모두 공백으로 대체하여 한 줄로 계속 흐르게 하는 스트리머"""
    def on_finalized_text(self, text: str, stream_end: bool = False):
        clean_text = text.replace("\n", " ").replace("\r", "")
        print(clean_text, end="", flush=True)
        if stream_end:
            print(" ", end="", flush=True)


def generate(tokenizer, model, prompt, temp=TEMPERATURE):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    streamer = NoNewlineStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    if SHOW_TEMP_INDICATOR:
        if temp < 1.4:
            color = "\033[92m"   # Green
        elif temp < 1.7:
            color = "\033[93m"   # Yellow
        else:
            color = "\033[91m"   # Red
        print(f"{color}[🌡️ {temp:.2f}]\033[0m ", end="", flush=True)

    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=temp,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY,
        streamer=streamer,
        pad_token_id=tokenizer.eos_token_id,
    )
    if TOP_K > 0:
        gen_kwargs["top_k"] = TOP_K

    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)

    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print("생성 함수 준비 완료")

## 7. 시작 시드 입력 (선택)
엔터만 누르면 랜덤 시드가 자동 생성됩니다.

In [ ]:
user_input = input("시작 시드 문장을 입력하세요 (엔터 입력 시 랜덤 생성): ").strip()
if user_input:
    prompt = user_input
else:
    prompt = random_seed(SEED_WORDS)
    print(f"랜덤 시드 사용: {prompt}")

## 8. 무한 개소리 루프 ▶️

이 셀을 실행하면 개소리가 흐르기 시작합니다.

- **종료:** Colab 의 ■ (셀 중지) 버튼을 누르세요.
- **새 시드로 다시 시작:** 위 7번 셀을 다시 실행한 뒤 이 셀을 다시 실행하세요.
- `MAX_ROUNDS` 를 양수로 두면 그 횟수만큼만 돌고 멈춥니다.

In [ ]:
print("로딩 완료. 개소리를 시작합니다. (■ 중지 버튼으로 종료)\n")

try:
    step = 0
    while True:
        if MAX_ROUNDS and step >= MAX_ROUNDS:
            print("\n\n[완료] 지정한 라운드 수에 도달했습니다.")
            break

        if USE_DYNAMIC_TEMP:
            theta = 2 * math.pi * step / DYNAMIC_TEMP_PERIOD
            temp = DYNAMIC_TEMP_MIN + (DYNAMIC_TEMP_MAX - DYNAMIC_TEMP_MIN) * (1 - math.cos(theta)) / 2
        else:
            temp = TEMPERATURE

        output = generate(tokenizer, model, prompt, temp=temp)   # 토큰 실시간 출력
        prompt = pick_words(output, FEEDBACK_WORDS)              # 출력 → 다음 시드
        step += 1

        if SLEEP_BETWEEN:
            time.sleep(SLEEP_BETWEEN)
except KeyboardInterrupt:
    print("\n\n[종료] 개소리 생성을 멈춥니다. 수고하셨습니다.")